# BrainMoE-PINN on Google Colab (GPU smoke)

Select a GPU runtime (T4 is sufficient for this bounded smoke). This notebook runs one real `BrainMoETrainer` update with the generic low-rank SDE and empirical CRPS. The batch is synthetic; this verifies runtime and optimizer wiring, not scientific performance.

Before running, create a source archive from the repository root and upload it to `/content/brain_moe_pinn_colab.tar.gz` using the Colab Files panel:

```bash
tar -czf brain_moe_pinn_colab.tar.gz --exclude=.git --exclude=.venv --exclude=.slim --exclude=__pycache__ --exclude=.pytest_cache --exclude='*.pyc' --exclude='*.pt' --exclude='*.pth' .
```

In [ ]:
from pathlib import Path
import sys
import tarfile

archive_path = Path('/content/brain_moe_pinn_colab.tar.gz')
assert archive_path.is_file(), 'Upload brain_moe_pinn_colab.tar.gz to /content first'
project_root = Path('/content/brain_moe_pinn')
project_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive_path, 'r:gz') as archive:
    archive.extractall(project_root, filter='data')
assert (project_root / 'train.py').is_file()
sys.path.insert(0, '/content')
print('PROJECT_ROOT', project_root)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime in Runtime > Change runtime type'
print('COLAB_RUNTIME', torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from torch.utils.data import DataLoader, Dataset

from brain_moe_pinn import BrainMoEPINN
from brain_moe_pinn.training.training_loop import BrainMoETrainer
from brain_moe_pinn.training.training_phases import (
    LossWeights,
    TrainingPhase,
    TrainingStage,
)

class OneBatch(Dataset):
    def __len__(self):
        return 1

    def __getitem__(self, index):
        return {
            'calcium': torch.randn(12, 64),
            'calcium_future': torch.randn(2, 12, 64),
            'dt': torch.tensor(1.0 / 64),
        }

torch.manual_seed(17)
model = BrainMoEPINN(
    eeg_channels=2,
    fmri_regions=3,
    latent_dim=8,
    use_neurostorm=False,
    use_kda_decoder=False,
    use_active_inference=False,
    use_species_conditioning=True,
    species='c_elegans',
    species_vocab=['c_elegans', 'human'],
    perturbation_dim=2,
    moe_num_shared=1,
    moe_num_routed=1,
    moe_top_k=1,
    generic_observation_only=True,
    transition_mode='sde',
    diffusion_rank=3,
    stochastic_samples=3,
)
head_before = model.transition_diffusion.rank_scale[-1].weight.detach().clone()
loader = DataLoader(OneBatch(), batch_size=1)
trainer = BrainMoETrainer(
    model,
    {
        'model_config': {'species': 'c_elegans'},
        'experiment_data': {
            'modalities': ['calcium'],
            'paired_next_step_targets': True,
            'future_steps': 2,
        },
        'generic_channels': 12,
        'generic_time': 64,
        'use_mixer': False,
    },
    log_dir='/content/brain_moe_pinn_colab_logs',
    checkpoint_dir='/content/brain_moe_pinn_colab_checkpoints',
    train_dataloader=loader,
)
phase = TrainingPhase(
    name='colab_sde_crps_smoke',
    stage=TrainingStage.STAGE_1_P1,
    total_steps=1,
    learning_rate=1e-4,
    warmup_steps=0,
    batch_size=1,
    rollout_steps=2,
    loss_weights=LossWeights(
        recon_eeg=0.0,
        recon_fmri=0.0,
        recon_meg=0.0,
        forecast_crps=1.0,
        forecast_horizon_weights=(1.0, 1.0),
    ),
)
trainer.train_phase(phase.to_runtime_config())
head_after = model.transition_diffusion.rank_scale[-1].weight.detach()
weight_delta = float((head_after - head_before.to(head_after.device)).abs().sum().item())
assert torch.isfinite(head_after).all()
assert weight_delta > 0.0, 'CRPS training step did not update the diffusion head'
print('COLAB_SDE_CRPS_TRAIN_STEP_OK', weight_delta)